# `GEA` tutorial: Gene expression dataset on Inflammatory Bowel Disease cohort

This notebook is used as a tutorial for creating sample-specific networks from gene expression data using a study from Inflammatory Bowel Disease (IBD) patients, and implementing Graph Explainable Attribution (GEA) to the learned embeddings from a Graph Neural Network model.

-----

**Data Description:**

- The dataset is composed of a study that reports the core ileal transcriptome in pediatric IBD patients and controls (BioProject accession: [PRJNA248469](https://www.ncbi.nlm.nih.gov/bioproject/248469)).
- Gene expression profiles from 303 samples of 204 Crohn's disease (CD) patients, 59 Ulcerative Colitis (UC) patients, and 40 healthy controls were analyzed.

**Goal:**

There are two main goals that follows this tutorial notebook: **(1)** generate sample-specific networks using a modified [LIONESS](https://link.springer.com/article/10.1186/s12885-019-6235-7) implementation, and **(2)** extract meaningful features from a trained GNN model using sparse autoencoders (SAE).

-----

## Loading IBD dataset

### Loading and filtering counts

`gea` has a series of functions to load all the data for constructing the sample-specific networks. Function `load_counts` at `dataloader` can be used to get a `pd.DataFrame` from a `.csv` file (or compatible):

In [3]:
from gea.dataloader import load_counts

# Load counts
counts_path = "data/tutorial-ibd-matrix.tsv"
raw_counts = load_counts(counts_path, delim = "\t", index_col = "Geneid")

# Evaluate
print(raw_counts.info())

<class 'pandas.core.frame.DataFrame'>
Index: 60664 entries, ENSG00000284662 to ENSG00000275405
Columns: 322 entries, SAMN03322967 to SAMN03323298
dtypes: int64(322)
memory usage: 149.5+ MB
None


We can see that the dataframe contains the Ensembl IDs for everything in the samples. We can use `ensembl_to_gene` from `utils` to get the IDs mapped to their corresponding gene symbol, as well as filtering other molecules (e.g. RNAs, pseudo-genes, non-coding genes, etc.):

In [4]:
from gea.utils import ensembl_to_gene

# Mapping Ensemlb IDs -> protein-coding genes (this may take a couple minutes)
gene_counts = ensembl_to_gene(raw_counts, species="human")

# Evaluate
print(f"After mapping {raw_counts.shape[0]} Ensembl IDs, only {gene_counts.shape[0]} genes remain.")

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
35 input query terms found dup hits:	[('ENSG00000228044', 2), ('ENSG00000226506', 2), ('ENSG00000261600', 2), ('ENSG00000234162', 2), ('E
1209 input query terms found no hit:	['ENSG00000272482', 'ENSG00000224621', 'ENSG00000234166', 'ENSG00000225643', 'ENSG00000287400', 'ENS


After mapping 60664 Ensembl IDs, only 19465 genes remain.


We can proceed to filter counts based on variance and abundance using the function `filter_genes` from `preprocessing`:

In [13]:
from gea.preprocessing import filter_genes

filter_counts = filter_genes(gene_counts, cpm_threshold=1, min_frac=0.1, pct=0.1)

print(f"Filtered from {gene_counts.shape[0]} to {filter_counts.shape[0]} genes.")

Filtered from 19465 to 1511 genes.


### Loading metadata

We can use `load_metadata` from `dataloader` to get all relevant information from the experiments In this case, we have special interest in the cohorts of patients with IBD and our column of interest in this data would be `source_name`:

In [14]:
from gea.dataloader import load_metadata

# Load metadata
metadata_path = "data/tutorial-ibd-metadata.txt"
metadata = load_metadata(metadata_path, delim = ",", index_col=None)

# Evaluate
print(metadata.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 322 entries, 0 to 321
Data columns (total 34 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Run                  322 non-null    object 
 1   age_at_diagnosis     322 non-null    float64
 2   Assay Type           322 non-null    object 
 3   AvgSpotLen           322 non-null    int64  
 4   Bases                322 non-null    int64  
 5   BioProject           322 non-null    object 
 6   BioSample            322 non-null    object 
 7   Bytes                322 non-null    int64  
 8   Center Name          322 non-null    object 
 9   Consent              322 non-null    object 
 10  DATASTORE filetype   322 non-null    object 
 11  DATASTORE provider   322 non-null    object 
 12  DATASTORE region     322 non-null    object 
 13  deep_ulcer           218 non-null    object 
 14  diagnosis            322 non-null    object 
 15  Experiment           322 non-null    obj

### Loading Protein-Protein Interaction network

As part of the modified LIONESS framework, previously known information about gene co-expression is given in the form of a Protein-Protein Interaction (PPI) network. For this, `load_ppi_network` from `dataloader` can be used to extract a PPI network from (STRING)[https://string-db.org] given a list of genes:

In [15]:
from gea.dataloader import load_ppi_network
from gea.utils import get_gene_list

# Get gene list from filtered data
gene_list = get_gene_list(filter_counts)

# Get PPI network
ppi_network = load_ppi_network(gene_list, species=9606)

# Evaluate
print(ppi_network.info())

Successfully retrieved PPI network!
Found 7829 interactions.
             stringId_A            stringId_B preferredName_A preferredName_B  \
0  9606.ENSP00000001146  9606.ENSP00000456609         CYP26B1           STRA6   
1  9606.ENSP00000001146  9606.ENSP00000310721         CYP26B1          CYP7B1   
2  9606.ENSP00000001146  9606.ENSP00000337224         CYP26B1            LRAT   
3  9606.ENSP00000001146  9606.ENSP00000320401         CYP26B1         UGT2B17   
4  9606.ENSP00000001146  9606.ENSP00000251566         CYP26B1          UGT2A3   

   ncbiTaxonId  score  nscore  fscore  pscore  ascore  escore  dscore  tscore  
0         9606  0.654     0.0     0.0   0.000   0.045   0.054    0.00   0.647  
1         9606  0.660     0.0     0.0   0.457   0.051   0.049    0.00   0.388  
2         9606  0.681     0.0     0.0   0.000   0.086   0.000    0.00   0.665  
3         9606  0.687     0.0     0.0   0.000   0.067   0.000    0.67   0.065  
4         9606  0.687     0.0     0.0   0.000   0.06

Now that we are using the PPI network as prior, we will prefer to only retain genes that appear on it. To do so, we can use `filter_ppi_nodes` in `preprocessing`:

In [19]:
from gea.preprocessing import filter_ppi_nodes

filter_counts_ppi = filter_ppi_nodes(filter_counts, ppi_network)

print(f"From {filter_counts.shape[0]} genes on the filtered counts dataframe, only {filter_counts_ppi.shape[0]} appear in the PPI network.")

From 1511 genes on the filtered counts dataframe, only 1165 appear in the PPI network.


### Loading Geneformer model

Usually Graph Neural Networks have no information on what exactly the node is if given without a flag. To overcome this, we are giving the model information of each gene by adding the Geneformer embedding as a feature in each node. We can load the Geneformer model by using `load_geneformer` from `dataloader`:

In [21]:
from gea.dataloader import load_geneformer

# Loading model and vocabulary
gf_model, gf_vocab = load_geneformer()

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ctheodoris/Geneformer
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Becayse we already have our counts dataframe at the last filtering step, we can call again `get_gene_list` from `utils` to get the embeddings we wish to extract from the model. To do so, `get_geneformer_embeddings` from `utils` is employed (the function randomize embeddings for genes that aren't found in model):

In [22]:
from gea.utils import get_geneformer_embeddings

# Get final gene list
gene_list = get_gene_list(filter_counts_ppi)

# Get Geneformer embeddings
gf_embeddings = get_geneformer_embeddings(gf_model, gf_vocab, gene_list)

Extracting Geneformer embeddings: 100%|██████████| 1165/1165 [00:00<00:00, 2377.13it/s]

Found embeddings for 1154/1165 genes.


## Creating sample-specific networks

### Normalizing counts

### Gene co-expression matrix

### Z-space constrained LIONESS

### Convert networks to PyG objects

## Training GNN model

## GEA: Graph Explainable Attribution with Sparse Autoencoder

### Training graph-level SAE